<a href="https://colab.research.google.com/github/diegocortes30/olist_ecommerce_analysis/blob/main/02_sql_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-commerce Analysis: 02 SQL Layer

**Goal:** answer five business questions about the Olist Brazilian e-commerce dataset using SQL (SQLite) from Python.

**Data:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle), 9 CSV files. Table structure, keys and data quality issues are documented in `01_data_understanding.ipynb`.

**Contents**
1. Setup and database creation
2. Join validation
3. Business questions (Q1 to Q5)
4. Appendix: SQL practice
5. Export queries to `.sql` files

### Assumptions used throughout
- Revenue = sum of `price` (shipping excluded).
- Orders with status `canceled` or `unavailable` are excluded from revenue analysis, but not from the cancellation analysis (Q4).
- Time series (Q3 and Q4 by month) cover 2017-01 to 2018-08, the months with complete data.
- Reviews of the same order are averaged into one score per order.
- An order counts once in each product category it contains.

## 1. Setup and database creation

SQLite is included with Python, so no server is needed. The database is stored as a file in Google Drive so it persists after the Colab session ends. The CSV files are loaded only the first time; later runs reuse the existing database.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sqlite3
import pandas as pd

RAW = '/content/drive/MyDrive/Olist/raw'
DB_PATH = '/content/drive/MyDrive/Olist/olist.db'

Mounted at /content/drive


In [2]:
db_exists = os.path.exists(DB_PATH)
conn = sqlite3.connect(DB_PATH)

if not db_exists:
    for file in os.listdir(RAW):
        if file.endswith('.csv'):
            name = file.replace('olist_', '').replace('_dataset.csv', '').replace('.csv', '')
            df = pd.read_csv(f'{RAW}/{file}')
            df.to_sql(name, conn, if_exists='replace', index=False)
            print(f'{name}: {len(df):,} rows loaded')
else:
    print('Database already exists, skipping CSV load.')

Database already exists, skipping CSV load.


In [3]:
def query(sql):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn)

# Check that the 9 tables exist
query("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,products
7,sellers
8,product_category_name_translation


## 2. Join validation

Before answering the business questions, we check how joins behave in this dataset. An order can have several items, payments or reviews, so joining tables can multiply rows. Knowing this prevents double counting.

In [4]:
# INNER JOIN: only orders that have items
query("""
SELECT COUNT(*) AS rows_inner
FROM orders AS o
JOIN order_items AS oi
  ON o.order_id = oi.order_id
""")

,rows_inner
0,112650


In [5]:
# LEFT JOIN: keeps every order, even those without items
query("""
SELECT COUNT(*) AS rows_left
FROM orders AS o
LEFT JOIN order_items AS oi
  ON o.order_id = oi.order_id
""")

,rows_left
0,113425


In [6]:
# Rows versus distinct orders after the join
query("""
SELECT COUNT(*) AS rows_count,
       COUNT(DISTINCT o.order_id) AS distinct_orders
FROM orders AS o
LEFT JOIN order_items AS oi
  ON o.order_id = oi.order_id
""")

,rows_count,distinct_orders
0,113425,99441


In [7]:
# Every order has a customer record
query("""
SELECT COUNT(*) AS rows_count
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id
""")

,rows_count
0,99441


**Result**
- The inner join returns 112,650 rows, the same number of rows as `order_items`, so every item belongs to an existing order.
- The left join returns 113,425 rows: 112,650 items plus 775 orders without items (canceled, unavailable or not yet processed orders).
- Those rows belong to 99,441 distinct orders, so an order with several items appears several times.
- Joining `orders` with `customers` returns 99,441 rows: every order has a customer record.


## 3. Business questions

| # | Question | Metric |
|---|---|---|
| Q1 | Which product categories receive the best and worst review scores? | Average review score per category (at least 100 reviewed orders) |
| Q2 | Which customer states generate the most revenue? | Sum of `price` per state |
| Q3 | How do monthly orders and revenue evolve over time? | Orders and sum of `price` per month |
| Q4 | What share of orders is canceled, and does it vary by state or month? | Canceled orders / total orders |
| Q5 | Which payment methods are used most, and how many installments do customers choose? | Payments per method; distribution of installments |

### Q1. Which product categories receive the best and worst review scores?

**Metric:** average review score per category, keeping only categories with at least 100 reviewed orders.

**Tables:** `order_reviews`, `order_items`, `products`, `product_category_name_translation`.

**Method:** two CTEs. `order_scores` gives one average score per order (some orders have several reviews). `order_categories` lists the categories of each order, translated to English, with `unknown` for products without category. Joining both avoids counting repeated reviews twice.

**Hypothesis (before running the query):** appliance categories receive the worst reviews.

In [8]:
q1_ctes = """
-- Q1: Which product categories receive the best and worst review scores?
-- Assumptions: reviews of the same order are averaged; an order counts once in each category it contains;
-- products without a category are labeled 'unknown'.
WITH order_scores AS (
    SELECT order_id,
           ROUND(AVG(review_score), 2) AS avg_review_score
    FROM order_reviews
    GROUP BY order_id
),
order_categories AS (
    SELECT DISTINCT oi.order_id,
           COALESCE(t.product_category_name_english,
                    p.product_category_name,
                    'unknown') AS category
    FROM order_items AS oi
    JOIN products AS p
      ON oi.product_id = p.product_id
    LEFT JOIN product_category_name_translation AS t
      ON p.product_category_name = t.product_category_name
)
"""

# Context: overall score and share of low-score orders
q1_context = q1_ctes + """
SELECT COUNT(*) AS reviewed_orders,
       ROUND(AVG(avg_review_score), 2) AS overall_avg,
       SUM(CASE WHEN avg_review_score <= 2 THEN 1 ELSE 0 END) AS low_score_orders,
       ROUND(100.0 * SUM(CASE WHEN avg_review_score <= 2 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_low_score
FROM order_scores
"""
query(q1_context)

,reviewed_orders,overall_avg,low_score_orders,pct_low_score
0,98673,4.09,14449,14.64


In [9]:
# Validation: order-category pairs, orders with items and distinct categories
q1_check = q1_ctes + """
SELECT COUNT(*) AS order_category_pairs,
       COUNT(DISTINCT order_id) AS orders,
       COUNT(DISTINCT category) AS categories
FROM order_categories
"""
query(q1_check)

,order_category_pairs,orders,categories
0,99470,98666,74


In [10]:
# Main query: average score per category
q1 = q1_ctes + """
SELECT oc.category,
       COUNT(*) AS reviewed_orders,
       ROUND(AVG(os.avg_review_score), 2) AS avg_score
FROM order_categories AS oc
JOIN order_scores AS os
  ON oc.order_id = os.order_id
GROUP BY oc.category
HAVING COUNT(*) >= 100
ORDER BY avg_score ASC
"""
q1_df = query(q1)
print(f'{len(q1_df)} categories with at least 100 reviewed orders')

# 10 worst categories
q1_df.head(10)

52 categories with at least 100 reviewed orders


,category,reviewed_orders,avg_score
0,office_furniture,1263,3.62
1,fashion_male_clothing,111,3.70
2,audio,347,3.83
3,construction_tools_safety,166,3.85
4,home_confort,395,3.86
5,fixed_telephony,214,3.90
6,unknown,1439,3.91
7,fashion_underwear_beach,120,3.93
8,bed_bath_table,9313,3.97
9,home_construction,487,3.97


In [11]:
# 10 best categories
q1_df.tail(10).iloc[::-1]

,category,reviewed_orders,avg_score
51,books_general_interest,508,4.46
50,books_technical,257,4.40
49,food_drink,226,4.38
48,luggage_accessories,1030,4.33
47,food,445,4.28
46,stationery,2295,4.24
45,pet_shop,1701,4.24
44,fashion_shoes,236,4.22
43,perfumery,3150,4.20
42,toys,3853,4.19


In [12]:
# Check of the hypothesis: appliance categories
q1_appliances = q1_ctes + """
SELECT oc.category,
       COUNT(*) AS reviewed_orders,
       ROUND(AVG(os.avg_review_score), 2) AS avg_score
FROM order_categories AS oc
JOIN order_scores AS os
  ON oc.order_id = os.order_id
WHERE oc.category LIKE '%appliances%'
GROUP BY oc.category
ORDER BY avg_score
"""
query(q1_appliances)

,category,reviewed_orders,avg_score
0,home_appliances_2,232,4.13
1,home_appliances,761,4.16
2,small_appliances,624,4.17
3,small_appliances_home_oven_and_coffee,75,4.29


**Result**
- 98,673 orders have at least one review. The average score is 4.09 and 14.6% of the orders score 2 or lower.
- Worst categories: `office_furniture` (3.62), `fashion_male_clothing` (3.70) and `audio` (3.83). The rest of the worst ten fall between 3.85 and 3.97.
- Best categories: `books_general_interest` (4.46), `books_technical` (4.40) and `food_drink` (4.38).
- `unknown` (products without category) averages 3.91 over 1,439 orders; it is not a real category.
- **Hypothesis not supported:** the four appliance categories score between 4.13 and 4.29, above the overall average.

**Finding (Q1):** review scores differ little across categories, but `office_furniture` stands out with the lowest score and a large sample. A low score can come from the product, the delivery or the seller; this dataset cannot separate them.

### Q2. Which customer states generate the most revenue?

**Metric:** sum of `price` per customer state, with the share of total revenue and the revenue per order.

**Tables:** `order_items`, `orders`, `customers`.

**Assumptions:** shipping is excluded and `canceled` and `unavailable` orders are not counted.

**Hypothesis (before running the query):** bigger states generate more revenue.

In [13]:
q2 = """
-- Q2: Which customer states generate the most revenue?
-- Assumptions: revenue = sum of price (shipping excluded); canceled and unavailable orders are excluded.
SELECT c.customer_state,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(oi.price), 2) AS revenue,
       ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS revenue_per_order,
       ROUND(100.0 * SUM(oi.price) / SUM(SUM(oi.price)) OVER (), 2) AS revenue_share_pct
FROM order_items AS oi
JOIN orders AS o
  ON oi.order_id = o.order_id
JOIN customers AS c
  ON o.customer_id = c.customer_id
WHERE o.order_status NOT IN ('unavailable', 'canceled')
GROUP BY c.customer_state
ORDER BY revenue DESC
"""
query(q2)

,customer_state,orders,revenue,revenue_per_order,revenue_share_pct
0,SP,41125,5163867.22,125.57,38.27
1,RJ,12697,1811623.42,142.68,13.43
2,MG,11496,1573508.20,136.87,11.66
3,RS,5415,742559.78,137.13,5.50
4,PR,4982,676883.06,135.87,5.02
5,SC,3599,518578.28,144.09,3.84
6,BA,3344,507108.83,151.65,3.76
7,DF,2120,300886.45,141.93,2.23
8,GO,1998,287870.46,144.08,2.13
9,ES,2018,273532.13,135.55,2.03


**Result**
- São Paulo (SP) generates 5.16 million in revenue, 38.3% of the total. Rio de Janeiro (RJ) follows with 1.81 million (13.4%) and Minas Gerais (MG) with 1.57 million (11.7%).
- The top three states (SP, RJ, MG) account for 63.4% of revenue; the top five (adding RS and PR) for 73.9%.
- The state with the most revenue does not have the highest order value: SP has the lowest revenue per order (125.57), while small states such as PB (216.34), AP (198.15) and AC (197.32) are above 195.
- Small states have few orders (for example, RR has 45), so their revenue per order is less reliable.

**Finding (Q2):** revenue is highly concentrated in the south-east. Large states win on volume, not on order value.

### Q3. How do monthly orders and revenue evolve over time?

**Metric:** number of orders and sum of `price` per month, plus revenue per order.

**Tables:** `orders`, `order_items`.

**Assumptions:** `canceled` and `unavailable` orders are excluded. First, a data coverage check shows which months are complete.

**Hypothesis (before running the query):** revenue follows the same peak as the number of orders.

In [14]:
# Data coverage: date range of the orders
query("""
SELECT MIN(order_purchase_timestamp) AS first_order,
       MAX(order_purchase_timestamp) AS last_order
FROM orders
""")

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [15]:
# Data coverage: orders per month, all months
query("""
SELECT strftime('%Y-%m', order_purchase_timestamp) AS year_month,
       COUNT(*) AS orders
FROM orders
GROUP BY year_month
ORDER BY year_month
""")

,year_month,orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


**Coverage check:** the data goes from 2016-09 to 2018-10, but 2016 has almost no orders (4, 324 and 1 in the three months that appear) and 2018-09 and 2018-10 have only 16 and 4. These months are incomplete, so the trend is analyzed only from **2017-01 to 2018-08**.

In [16]:
q3 = """
-- Q3: How do monthly orders and revenue evolve over time?
-- Assumptions: months from 2017-01 to 2018-08 (complete months); canceled and unavailable orders excluded;
-- revenue = sum of price.
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS year_month,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(oi.price), 2) AS revenue,
       ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS revenue_per_order
FROM orders AS o
JOIN order_items AS oi
  ON o.order_id = oi.order_id
WHERE o.order_status NOT IN ('unavailable', 'canceled')
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp < '2018-09-01'
GROUP BY year_month
ORDER BY year_month
"""
query(q3)

,year_month,orders,revenue,revenue_per_order
0,2017-01,787,120098.27,152.60
1,2017-02,1718,244959.35,142.58
2,2017-03,2617,368341.32,140.75
3,2017-04,2377,353842.98,148.86
4,2017-05,3640,503159.19,138.23
5,2017-06,3205,429916.61,134.14
6,2017-07,3946,492287.30,124.76
7,2017-08,4272,568245.79,133.02
8,2017-09,4227,621415.91,147.01
9,2017-10,4547,660179.62,145.19


**Result**
- Orders and revenue grow sharply during 2017: from 787 orders and 120 thousand in January to 4,547 orders and 660 thousand in October.
- The peak is **2017-11**, with 7,421 orders and 1.00 million in revenue.
- During 2018 the monthly volume stays between about 6,100 and 7,200 orders and between about 840 thousand and 990 thousand in revenue.
- Revenue per order stays in a narrow range (roughly 125 to 153), so **revenue follows order volume**. The hypothesis is supported.
- A possible explanation for the November 2017 peak is Black Friday, but this analysis does not test it.

**Finding (Q3):** the business grew quickly during 2017 and stabilized in 2018. Revenue growth comes from more orders, not from more expensive orders.

### Q4. What share of orders is canceled, and does it vary by state or month?

**Metric:** canceled orders / total orders, overall, by customer state and by month.

**Tables:** `orders`, `customers`.

**Assumptions:** all orders count in the denominator. States and months with fewer than 100 orders are left out, because one cancellation would move the percentage too much. The monthly view covers 2017-01 to 2018-08, as in Q3.

**Hypothesis (before running the query):** months with more orders have more cancellations.

In [17]:
q4_overall = """
-- Q4: overall cancellation rate
SELECT COUNT(*) AS total_orders,
       SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
       ROUND(100.0 * SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS canceled_pct
FROM orders AS o
"""
query(q4_overall)

,total_orders,canceled_orders,canceled_pct
0,99441,625,0.63


In [18]:
q4_by_state = """
-- Q4: cancellation rate by customer state (states with at least 100 orders)
SELECT c.customer_state,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
       ROUND(100.0 * SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS canceled_pct
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id
GROUP BY c.customer_state
HAVING COUNT(*) >= 100
ORDER BY canceled_pct DESC
"""
query(q4_by_state)

,customer_state,total_orders,canceled_orders,canceled_pct
0,RO,253,3,1.19
1,PI,495,4,0.81
2,SP,41746,327,0.78
3,RJ,12852,86,0.67
4,GO,2020,13,0.64
5,MG,11635,64,0.55
6,MA,747,4,0.54
7,SC,3637,19,0.52
8,CE,1336,7,0.52
9,BA,3380,16,0.47


In [19]:
q4_by_month = """
-- Q4: cancellation rate by month (2017-01 to 2018-08, months with at least 100 orders)
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS year_month,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
       ROUND(100.0 * SUM(CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS canceled_pct
FROM orders AS o
WHERE o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp < '2018-09-01'
GROUP BY year_month
HAVING COUNT(*) >= 100
ORDER BY year_month
"""
query(q4_by_month)

,year_month,total_orders,canceled_orders,canceled_pct
0,2017-01,800,3,0.38
1,2017-02,1780,17,0.96
2,2017-03,2682,33,1.23
3,2017-04,2404,18,0.75
4,2017-05,3700,29,0.78
5,2017-06,3245,16,0.49
6,2017-07,4026,28,0.70
7,2017-08,4331,27,0.62
8,2017-09,4285,20,0.47
9,2017-10,4631,26,0.56


**Result**
- Only 0.63% of orders are canceled (625 of 99,441).
- By state, the rate goes from 0.00% to 1.19% among the 24 states with at least 100 orders. RO has the highest rate, but it comes from 3 canceled orders out of 253. SP has 0.78%, with the largest number of cancellations (327).
- By month (2017-01 to 2018-08) the rate goes from 0.19% (2017-12) to 1.29% (2018-08). The highest values are 2017-03 (1.23%), 2018-02 (1.09%) and 2018-08 (1.29%).
- **Hypothesis not supported:** November 2017, the month with the most orders, has 0.49%, below the overall rate.

**Finding (Q4):** cancellations are rare and do not follow demand. Differences between states are small and based on few cases. The data does not explain the monthly peaks.

### Q5. Which payment methods are used most, and how many installments do customers choose?

**Metric:** number of payments (and distinct orders) per `payment_type`; distribution of `payment_installments`; average and maximum installments per method.

**Tables:** `order_payments`.

**Assumptions:** an order can have several payments, so payments and orders are counted separately. Payments with a value of 0 or 0 installments are shown for reference and flagged as a data quality issue.

**Hypothesis (before running the query):** credit card is the most used method and the one that allows the most installments.

In [20]:
q5_payment_methods = """
-- Q5: payments and orders per payment method
SELECT payment_type,
       COUNT(*) AS payments,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS payments_pct
FROM order_payments
GROUP BY payment_type
ORDER BY payments DESC
"""
query(q5_payment_methods)

,payment_type,payments,orders,payments_pct
0,credit_card,76795,76505,73.92
1,boleto,19784,19784,19.04
2,voucher,5775,3866,5.56
3,debit_card,1529,1528,1.47
4,not_defined,3,3,0.00


In [21]:
q5_installments = """
-- Q5: distribution of installments
SELECT payment_installments,
       COUNT(*) AS payments,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(AVG(payment_value), 2) AS avg_value,
       MIN(payment_value) AS min_value,
       MAX(payment_value) AS max_value
FROM order_payments
GROUP BY payment_installments
ORDER BY payment_installments
"""
query(q5_installments)

,payment_installments,payments,orders,avg_value,min_value,max_value
0,0,2,2,94.31,58.69,129.94
1,1,52546,49060,112.42,0.00,13664.08
2,2,12413,12389,127.23,20.03,2442.82
3,3,10461,10443,142.54,30.00,3256.14
4,4,7098,7088,163.98,40.00,3126.50
5,5,5239,5234,183.47,49.99,4809.44
6,6,3920,3916,209.85,59.99,2828.42
7,7,1626,1623,187.67,69.97,2094.44
8,8,4268,4253,307.74,37.58,6929.31
9,9,644,644,203.44,90.11,2116.01


In [22]:
q5_by_method = """
-- Q5: average and maximum installments per payment method
SELECT payment_type,
       COUNT(*) AS payments,
       ROUND(AVG(payment_installments), 2) AS avg_installments,
       MAX(payment_installments) AS max_installments
FROM order_payments
GROUP BY payment_type
ORDER BY payments DESC
"""
query(q5_by_method)

,payment_type,payments,avg_installments,max_installments
0,credit_card,76795,3.51,24
1,boleto,19784,1.00,1
2,voucher,5775,1.00,1
3,debit_card,1529,1.00,1
4,not_defined,3,1.00,1


In [23]:
q5_anomalies = """
-- Q5: payments with 0 installments or a value of 0 (data quality check)
SELECT *
FROM order_payments
WHERE payment_installments = 0 OR payment_value = 0
"""
query(q5_anomalies)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
1,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
2,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
3,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
4,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
5,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
6,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
7,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
8,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
9,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00


**Result**
- Credit card is the most used method: 73.9% of payments (76,795) and 76,505 orders. Boleto follows with 19.0%, voucher with 5.6% and debit card with 1.5%.
- Vouchers appear in 3,866 orders but account for 5,775 payments, so one order can use several vouchers.
- **Only credit card payments use installments:** average of 3.51 and up to 24. Boleto, voucher, debit card and `not_defined` always have 1 installment.
- Half of all payments (52,546 of 103,886, 50.6%) are made in a single installment. The most common installment counts are 1, 2, 3 and 4, with a smaller peak at 10.
- The average payment value grows with the number of installments (112 with 1 installment, 415 with 10), which suggests that more expensive purchases are paid in installments.
- **Data quality:** 11 payments are anomalous. 9 have a value of 0 (6 vouchers and the 3 `not_defined` rows) and 2 credit card payments have 0 installments.

**Hypothesis supported:** credit card is both the most used method and the only one that allows installments.

**Finding (Q5):** credit card dominates payments, and installments are a credit card feature. Zero-value and zero-installment payments should be excluded from payment analysis.

## 4. Appendix: SQL practice

Basic queries written while learning SQL (`SELECT`, `WHERE`, `GROUP BY`, aggregate functions and `strftime`). They do not answer the business questions, but they document the learning process.

In [24]:
# SELECT: first rows of a table
query("""
SELECT *
FROM orders
LIMIT 5
""")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [25]:
# SELECT: choose columns
query("""
SELECT order_id, order_status
FROM orders
LIMIT 10
""")

,order_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered
5,a4591c265e18cb1dcee52889e2d8acc3,delivered
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced
7,6514b8ad8028c9f2cc2374ded245783f,delivered
8,76c6e866289321a7c93b82b54852dc33,delivered
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered


In [26]:
# GROUP BY: orders per status
query("""
SELECT order_status, COUNT(*) AS orders_count
FROM orders
GROUP BY order_status
ORDER BY orders_count DESC
""")

,order_status,orders_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [27]:
# WHERE: delivered orders
query("""
SELECT COUNT(*) AS delivered_orders
FROM orders
WHERE order_status = 'delivered'
""")

,delivered_orders
0,96478


In [28]:
# Aggregate functions: summary of item prices
query("""
SELECT COUNT(*) AS items,
       SUM(price) AS total_price,
       AVG(price) AS avg_price,
       MIN(price) AS min_price,
       MAX(price) AS max_price
FROM order_items
""")

,items,total_price,avg_price,min_price,max_price
0,112650,13591643.7,120.653739,0.85,6735.0


In [29]:
# WHERE: canceled orders
query("""
SELECT COUNT(*) AS canceled_orders
FROM orders
WHERE order_status = 'canceled'
""")

,canceled_orders
0,625


In [30]:
# WHERE with OR: canceled or unavailable orders
query("""
SELECT COUNT(*) AS unavailable_and_canceled_orders
FROM orders
WHERE order_status = 'canceled'
      OR order_status = 'unavailable'
""")

,unavailable_and_canceled_orders
0,1234


In [31]:
# WHERE with a number: items priced above 1,000
query("""
SELECT COUNT(*) AS products_above_1000
FROM order_items
WHERE price > 1000
""")

,products_above_1000
0,844


In [32]:
# JOIN: orders with the state of their customer
query("""
SELECT o.order_id, o.order_status, c.customer_state
FROM orders AS o
JOIN customers AS c
  ON o.customer_id = c.customer_id
LIMIT 5
""")

,order_id,order_status,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,BA
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,SP


In [33]:
# strftime: extract year and month from a date stored as text
query("""
SELECT order_purchase_timestamp,
       strftime('%Y-%m', order_purchase_timestamp) AS year_month
FROM orders
LIMIT 5
""")

,order_purchase_timestamp,year_month
0,2017-10-02 10:56:33,2017-10
1,2018-07-24 20:41:37,2018-07
2,2018-08-08 08:38:49,2018-08
3,2017-11-18 19:28:06,2017-11
4,2018-02-13 21:18:39,2018-02


## 5. Export queries to `.sql` files

Each analysis query is saved as a `.sql` file so it can be reused outside the notebook and shown in the `sql/` folder of the repository.

In [34]:
sql_dir = '/content/drive/MyDrive/Olist/sql'
os.makedirs(sql_dir, exist_ok=True)

queries = {
    'q1_category_reviews': q1,
    'q1_appliances_check': q1_appliances,
    'q2_revenue_by_state': q2,
    'q3_monthly_orders_revenue': q3,
    'q4_cancellation_overall': q4_overall,
    'q4_cancellation_by_state': q4_by_state,
    'q4_cancellation_by_month': q4_by_month,
    'q5_payment_methods': q5_payment_methods,
    'q5_installments_distribution': q5_installments,
    'q5_installments_by_method': q5_by_method,
    'q5_zero_value_payments': q5_anomalies,
}

for name, sql in queries.items():
    with open(f'{sql_dir}/{name}.sql', 'w') as f:
        f.write(sql.strip() + '\n')

sorted(os.listdir(sql_dir))

['q1_appliances_check.sql',
 'q1_category_reviews.sql',
 'q2_revenue_by_state.sql',
 'q3_monthly_orders_revenue.sql',
 'q4_cancellation_by_month.sql',
 'q4_cancellation_by_state.sql',
 'q4_cancellation_overall.sql',
 'q5_installments_by_method.sql',
 'q5_installments_distribution.sql',
 'q5_payment_methods.sql',
 'q5_zero_value_payments.sql']

### Limitations
- Review scores are averaged per order, so differences between the reviews of one order are not visible.
- A category's score mixes product, delivery and seller effects.
- Revenue excludes shipping and does not account for returns or refunds.